# TFM · MatterGen · 100 estructuras DESDE CERO · v4.8 — RUN ALL

**Este notebook está pensado para un solo clic en `Run All`, dentro de una Draft Session activa.**

He eliminado los respaldos remotos que impedían ejecutar las versiones 4.6 y 4.7.
**Solo necesita el secreto `MP_API_KEY`**. No necesita `KAGGLE_API_TOKEN` ni `KAGGLE_USERNAME`.

**Instrucciones sencillas:**

1. Importar el notebook a Kaggle, activar **GPU T4 x2** (también sirve una T4) e **Internet On**.
2. Activar `MP_API_KEY` en **Add-ons → Secrets**.
3. Iniciar la sesión *Draft* y esperar al punto verde.
4. Pulsar **Run All** una sola vez y esperar a que termine.
5. Buscar la carpeta `/kaggle/working/TFM_RESULTADOS` y descargar el archivo `/kaggle/working/Resultados_TFM_Kaggle_100.zip`.

**Aviso importante:** esta versión **no ofrece copias externas**. La carpeta de `/kaggle/working` pertenece a la sesión y se puede perder si Kaggle la reinicia. Se recomienda descargar el ZIP antes de cerrar la sesión.

Se planifican 100 intentos de generación (25 por familia química). No se considera demostrado que esos materiales sean electrolitos viables: las propiedades y los filtros deben evaluarse y documentarse.

## 0. Preparar la ejecución

Aquí fijo las rutas y leo la clave de Materials Project sin mostrarla en pantalla.

Para evitar mezclar resultados de distintas sesiones, si encuentro cristales antiguos me detengo. Si solo hay restos de un intento fallido con 0 CIF, los limpio y empiezo desde cero.

In [ ]:
from pathlib import Path
import os
import json
import shutil
from kaggle_secrets import UserSecretsClient

# Estas rutas son siempre las mismas; así evito NameError si el kernel arranca limpio.
TRABAJO = Path('/kaggle/working')
RESULTADOS = TRABAJO / 'TFM_RESULTADOS'
PROYECTO = TRABAJO / 'TFM_KAGGLE_100'

# No borro una generación anterior si realmente contiene CIF.
# A veces quedan carpetas después de un error; compruebo primero su contenido.
if RESULTADOS.exists():
    zips_cif = list(RESULTADOS.rglob('generated_crystals_cif.zip'))
    cif_sueltos = list(RESULTADOS.rglob('*.cif'))
    manifiesto = RESULTADOS / 'generacion' / 'manifest_generacion.json'
    n_previos = 0
    if manifiesto.exists():
        n_previos = int(json.loads(manifiesto.read_text(encoding='utf-8')).get('cif_generados_verificados', 0))
    if zips_cif or cif_sueltos or n_previos > 0:
        raise RuntimeError('Ya hay estructuras en esta Draft Session. Descárgalas antes de comenzar desde cero con una sesión nueva.')

# Eliminamos únicamente restos de un arranque fallido que no llegó a generar CIF.
# Nunca tocamos otras carpetas del usuario en Kaggle.
for carpeta in [RESULTADOS, PROYECTO, TRABAJO / 'tfm_envs']:
    if carpeta.exists():
        shutil.rmtree(carpeta)
(TRABAJO / 'Resultados_TFM_Kaggle_100.zip').unlink(missing_ok=True)
PROYECTO.mkdir(parents=True, exist_ok=True)
os.environ['MPLBACKEND'] = 'Agg'

# La clave se lee del gestor de secretos; jamás la escribo en el notebook ni en GitHub.
try:
    _clave = UserSecretsClient().get_secret('MP_API_KEY')
except Exception as e:
    raise RuntimeError('Activa y autoriza MP_API_KEY en Kaggle → Add-ons → Secrets.') from e
if not _clave or not str(_clave).strip():
    raise RuntimeError('MP_API_KEY está vacío o no es accesible.')
os.environ['MP_API_KEY'] = str(_clave).strip()
del _clave
print('Preparación correcta: se empieza desde cero, sin respaldos remotos.')
print('Carpeta de resultados prevista:', RESULTADOS)

## 1. Escribir todos los scripts comentados del proyecto

Cada celda siguiente contiene un archivo real del código. Así el notebook se puede importar y ejecutar sin subir manualmente una carpeta de scripts.

He conservado explicaciones frecuentes y naturales en español: qué se hace, por qué se hace y qué significan las limitaciones. Los archivos JSON no admiten comentarios internos, pero sus parámetros están explicados en el README del proyecto.

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/config_tfm_kaggle.json
{
  "esquema": "TFM_KAGGLE_100_v4",
  "semilla": 42,
  "generacion": {
    "modelo": "chemical_system_energy_above_hull",
    "energia_objetivo_eV_atom": 0.05,
    "guidance": 2.0,
    "max_por_lote": 5,
    "generar_trayectorias": false,
    "sistemas": [
      {"sistema": "Li-P-S", "familia": "sulfuros", "ejemplo_referencia": "Li3PS4", "numero": 25},
      {"sistema": "Li-La-Zr-O", "familia": "oxidos_garnet", "ejemplo_referencia": "Li7La3Zr2O12 (LLZO)", "numero": 25},
      {"sistema": "Li-In-Cl", "familia": "haluros", "ejemplo_referencia": "Li3InCl6", "numero": 25},
      {"sistema": "Li-Al-Ti-P-O", "familia": "oxidos_NASICON", "ejemplo_referencia": "Li1+xAlxTi2-x(PO4)3 (LATP)", "numero": 25}
    ]
  },
  "datos": {
    "max_por_consulta": 200,
    "portadores": ["Li"]
  },
  "modelo_predictivo": {
    "targets": ["energy_above_hull", "band_gap"],
    "num_folds": 5,
    "max_filas_entrenamiento": 1800
  },
  "seleccion": {
    "min_band_gap_ev": null,
    "max_energy_above_hull_predicha_ev_atom": null,
    "pesos_exploratorios": {
      "energy_above_hull_predicha": 0.65,
      "band_gap_predicho_proxy": 0.25,
      "novedad_formula": 0.10
    },
    "pesos_integrales": null
  },
  "justificacion": {
    "tamano_muestra": "100 intentos de generacion: compromiso exploratorio y presupuesto GPU, no tamano muestral optimo estadistico",
    "sistemas": "Cuatro sistemas de electrolitos de litio documentados; la eleccion no equivale a demostrar aptitud funcional",
    "referencias": [
      "https://github.com/microsoft/mattergen",
      "https://doi.org/10.1016/j.jpowsour.2025.238150",
      "https://doi.org/10.1016/j.ensm.2018.02.020",
      "https://doi.org/10.1016/j.ensm.2026.105138"
    ]
  }
}

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/requirements_analisis.txt
# Librerías principales para leer y tratar los datos del experimento.
numpy>=1.26,<2
pandas>=2.1,<3

# Los modelos de regresión, métricas, PCA y K-Means están en scikit-learn.
scikit-learn>=1.4,<2

# Figuras PNG para la memoria; el backend Agg se configura en el código.
matplotlib>=3.8,<4

# Fuentes y análisis de materiales cristalinos. Materials Project exige API key.
mp-api>=0.45
pymatgen>=2024.6
matminer>=0.9

# Guardado de modelos, pruebas y generación de tablas legibles en Markdown.
joblib>=1.3
pytest>=8,<10
tabulate>=0.9

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/exportar_kaggle.py
#!/usr/bin/env python3
"""Exporta los resultados del TFM con una auditoría SHA256 clara y sin duplicados.

Este archivo no genera cristales ni inventa resultados: solo empaqueta lo que
existe en el directorio de resultados en el momento de llamar a ``pack``.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import zipfile
from datetime import datetime, timezone
from pathlib import Path


# Aquí dejo agrupados los nombres que nunca deberían acabar en un ZIP público.
# La clave de Materials Project se usa desde los secretos de Kaggle, no se copia.
NOMBRES_SENSIBLES = {'.env', 'MP_API_KEY', 'credentials.json'}
ARCHIVO_AUDITORIA = 'AUDITORIA_SHA256.json'


def sha256_bytes(data: bytes) -> str:
    """Calcula la huella de un fichero; basta un cambio para que sea diferente."""
    return hashlib.sha256(data).hexdigest()


def incluir_en_zip(fichero: Path, raiz: Path, destino: Path) -> bool:
    """Decide qué entra en el paquete y qué conviene dejar fuera."""
    if not fichero.is_file() or fichero.is_symlink():
        return False

    # Muy importante: no metemos el propio ZIP dentro de sí mismo.
    # Tampoco la auditoría antigua. Esta última era la causa del warning v4.4.
    if fichero.resolve() == destino.resolve():
        return False
    if fichero.name == ARCHIVO_AUDITORIA or fichero.name in NOMBRES_SENSIBLES:
        return False

    # No hace falta enviar cachés ni intentos incompletos. Se mantienen en
    # Kaggle para diagnosticar fallos, pero complicarían innecesariamente el ZIP.
    ruta_relativa = fichero.relative_to(raiz)
    partes = set(ruta_relativa.parts)
    if '__pycache__' in partes or '.ipynb_checkpoints' in partes:
        return False
    if any(parte.startswith('intento_incompleto_') for parte in partes):
        return False
    if fichero.name.endswith('.zip.tmp'):
        return False

    # MatterGen puede generar trayectorias muy grandes. Para las tablas del TFM
    # usamos las estructuras finales, que ya vienen en los ZIP de cada lote.
    if fichero.name in {'generated_crystals.extxyz', 'generated_trajectories.zip'}:
        return False
    if 'cif_validados' in partes and fichero.suffix.lower() == '.cif':
        return False
    return True


def pack(raiz, archivo_zip):
    """Crea una exportación nueva; se puede llamar varias veces sin duplicar."""
    root = Path(raiz)
    target = Path(archivo_zip)
    root.mkdir(parents=True, exist_ok=True)
    target.parent.mkdir(parents=True, exist_ok=True)

    # Primero inventariamos los resultados reales. La auditoría anterior queda
    # fuera siempre, así que nunca se calcula el hash de una auditoría caducada.
    items = []
    for child in sorted(root.rglob('*')):
        if not incluir_en_zip(child, root, target):
            continue
        contenido = child.read_bytes()
        items.append({
            'ruta': child.relative_to(root).as_posix(),
            'sha256': sha256_bytes(contenido),
            'bytes': len(contenido),
        })

    # La fecha es la de esta exportación. No se presenta como fecha de síntesis
    # ni como prueba de que todos los cálculos hayan terminado correctamente.
    audit = {
        'fecha_utc': datetime.now(timezone.utc).isoformat(),
        'n_archivos': len(items),
        'archivos': items,
        'nota': ('El ZIP contiene los resultados disponibles, incluso si la '
                 'ejecución es parcial. La auditoría no certifica validez científica.'),
    }
    audit_path = root / ARCHIVO_AUDITORIA
    audit_path.write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8')

    # Uso un archivo temporal: si Kaggle corta la exportación a mitad,
    # no sustituimos el último ZIP completo por otro incompleto.
    temporal = target.with_name(target.name + '.tmp')
    try:
        with zipfile.ZipFile(temporal, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for item in items:
                z.write(root / item['ruta'], arcname=item['ruta'])
            z.write(audit_path, arcname=ARCHIVO_AUDITORIA)

        # Una vez terminado, comprobamos tanto el ZIP como el inventario.
        # Así sabemos que no hemos vuelto a añadir dos veces el mismo nombre.
        with zipfile.ZipFile(temporal, 'r') as z:
            nombres = z.namelist()
            if len(nombres) != len(set(nombres)):
                raise RuntimeError('El ZIP contiene nombres repetidos')
            dañado = z.testzip()
            if dañado is not None:
                raise RuntimeError(f'Archivo comprimido dañado: {dañado}')
            for item in items:
                contenido = z.read(item['ruta'])
                if len(contenido) != item['bytes'] or sha256_bytes(contenido) != item['sha256']:
                    raise RuntimeError(f'Fallo en auditoría de: {item["ruta"]}')

        # Solo ahora hacemos el cambio definitivo. Un nuevo checkpoint pasa
        # a reemplazar al anterior, pero únicamente después de verificarlo.
        os.replace(temporal, target)
    finally:
        if temporal.exists():
            temporal.unlink()

    print(f'ZIP EXPORTADO: {target} '
          f'({target.stat().st_size / 1024 / 1024:.1f} MB, {len(items)} archivos)')
    return audit


def main():
    # Dejo esta interfaz para poder exportar también desde una terminal:
    # python exportar_kaggle.py --raiz resultados --zip resultados.zip
    parser = argparse.ArgumentParser()
    parser.add_argument('--raiz', required=True)
    parser.add_argument('--zip', required=True)
    args = parser.parse_args()
    pack(args.raiz, args.zip)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/generar_kaggle.py
#!/usr/bin/env python3
"""Generación verificable MatterGen (Kaggle), 100 muestras planeadas, por lotes.

No se crean CIF sintéticos ni se afirma éxito sin leer generated_crystals_cif.zip.
Cada lote usa semilla específica. Los lotes se reconocen dentro de una misma
sesión, pero el notebook v4.8 NO importa resultados de sesiones anteriores.
"""
from __future__ import annotations
import argparse, csv, hashlib, json, os, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

# Este nombre debe coincidir con el modelo preentrenado de MatterGen.
# No significa que estemos calculando experimentalmente la energía.
MODEL='chemical_system_energy_above_hull'

# Guardo la fecha en UTC para comparar ejecuciones aunque cambie la zona horaria.
def now(): return datetime.now(timezone.utc).isoformat()
# Los manifiestos se escriben primero en un temporal.
# Así, si se corta la sesión, es menos probable dejar un JSON a medias.
def write_json(p, v):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+'.tmp')
    tmp.write_text(json.dumps(v,indent=2,ensure_ascii=False,default=str),encoding='utf-8');tmp.replace(p)
# Antes de gastar tiempo de GPU compruebo que el plan tenga sentido.
# Aquí es donde se detectaría un reparto equivocado de las 100 muestras.
def load_config(path):
    config=json.loads(Path(path).read_text(encoding='utf-8'))
    if config['generacion']['modelo'] != MODEL: raise ValueError('Modelo MatterGen inesperado')
    systems=config['generacion']['sistemas']
    if not systems or len({s['sistema'] for s in systems})!=len(systems): raise ValueError('Sistemas no válidos o repetidos')
    for s in systems:
        if 'Li' not in s['sistema'].split('-') or int(s['numero'])<1: raise ValueError('Sistemas requieren Li y cuota >0')
    # La suma debe ser 100 exactamente: son intentos planificados, no 100 electrolitos válidos.
    if sum(int(s['numero']) for s in systems)!=100: raise ValueError('Se requieren exactamente 100 intentos (cuotas suman 100)')
    return config

# Divido el trabajo en lotes para poder recuperarlo si Kaggle se desconecta.
# Hacerlo todo de golpe sería arriesgar muchas horas de GPU.
def plan_batches(cfg):
    g=cfg['generacion']; batch_max=int(g['max_por_lote'])
    if batch_max<1: raise ValueError('max_por_lote debe ser >=1')
    result=[]
    # Cada familia recibe su cuota y cada lote una semilla distinta.
    # La semilla ayuda a repetir el ensayo, aunque CUDA no siempre es 100 % determinista.
    for sys_index,s in enumerate(g['sistemas']):
        left=int(s['numero']); batch_index=0
        while left:
            # Primer lote muy pequeño para detectar errores sin gastar una cuota grande.
            batch=(1 if sys_index==0 and batch_index==0 else min(batch_max,left))
            batch=min(batch,left)
            result.append({'sistema':s['sistema'],'familia':s['familia'],
                          'indice_sistema':sys_index,'indice_lote':batch_index+1,
                          'cantidad':batch,'semilla':int(cfg['semilla'])+sys_index*10000+batch_index,
                          'lote_id':f'{sys_index+1:02d}_{s["sistema"].replace("-","_")}_lote_{batch_index+1:02d}'})
            left-=batch;batch_index+=1
    if sum(p['cantidad'] for p in result)!=100: raise AssertionError('Recuento incorrecto')
    return result

# Aquí solo cuento CIF dentro de un ZIP que pueda leerse.
# La validez química y geométrica se comprobará después en el análisis.
def count_cif(archive):
    with zipfile.ZipFile(archive) as z:
        bad=z.testzip()
        if bad:raise ValueError(f'ZIP CIF dañado: {bad}')
        names=[s for s in z.namelist() if s.lower().endswith('.cif') and not s.endswith('/')]
        return len(names)

# Esta es la parte delicada: lanza MatterGen o recupera un lote ya hecho.
# Prefiero parar con un error a mezclar dos ejecuciones diferentes.
def run_one(entry, cfg, root, exe, command_runner=subprocess.run):
    dest=Path(root)/'lotes'/entry['lote_id']; dest.mkdir(parents=True,exist_ok=True)
    done=dest/'lote_finalizado.json'; arch=dest/'generated_crystals_cif.zip'
    # Si existe una marca de finalizado no me fío solo de ella: verifico el ZIP real.
    if done.exists():
        record=json.loads(done.read_text(encoding='utf-8'))
        if record.get('lote_id')==entry['lote_id'] and arch.exists() and count_cif(arch)==entry['cantidad']:
            return record
        raise RuntimeError(f'Manifiesto y CIF inconsistentes: {dest}; revisar manualmente')
    # Si Kaggle cortó la ejecución, puede haber archivos a medio escribir.
    # Los aparto antes de volver a intentar el lote; no los cuento como terminados.
    if any(dest.iterdir()):
        # Solo reintento permitido cuando NO hay un ZIP resultado real en directorio incompleto.
        if arch.exists():
            found=count_cif(arch)
            if found==entry['cantidad']:
                record={**entry,'estado':'OK_RECUPERADO','cif_generados':found,'fecha_fin_utc':now(),
                        'archivo_zip_cif':str(arch.relative_to(root))}
                write_json(done,record);return record
            raise RuntimeError(f'Lote parcial {dest} ({found} CIF); sin reintento automático para no mezclar muestras')
        # La ejecución interrumpida puede dejar extxyz / logs. Se separan y archivan
        # antes de relanzar; no se mezclan resultados de distintas semillas.
        old=dest/f'intento_incompleto_{int(time.time()*1000000)}'
        old.mkdir(exist_ok=True)
        for child in list(dest.iterdir()):
            if child != old:child.rename(old/child.name)
    # Estas son condiciones que pedimos al generador, no propiedades medidas.
    # Especialmente energy_above_hull=0,05 es un objetivo, no una prueba de estabilidad.
    cond={'chemical_system':entry['sistema'],
          'energy_above_hull':float(cfg['generacion']['energia_objetivo_eV_atom'])}
    # Uso el Python del entorno propio de MatterGen porque el kernel de Kaggle
    # y la librería CUDA necesitan dependencias diferentes.
    python_mattergen=Path(exe).parent/'python'
    if not python_mattergen.is_file():
        raise RuntimeError(f'No existe el intérprete MatterGen: {python_mattergen}')
    launcher=Path(__file__).with_name('lanzar_mattergen.py')
    if not launcher.is_file():
        raise RuntimeError(f'Falta el lanzador con control de semilla: {launcher}')
    # En la orden incluyo el sistema, el lote y la semilla.
    # Así cada generación queda identificada y se puede rastrear en el informe.
    cmd=[str(python_mattergen),str(launcher),
         '--output-path',str(dest),
         '--pretrained-name',MODEL,
         '--batch-size',str(entry['cantidad']),
         '--seed',str(entry['semilla']),
         '--conditioning-json',json.dumps(cond,ensure_ascii=False),
         '--guidance',str(float(cfg['generacion']['guidance'])),
         '--record-trajectories',str(bool(cfg['generacion'].get('generar_trayectorias',False))).lower()]
    (dest/'comando_sin_secretos.txt').write_text(' '.join(cmd),encoding='utf-8')
    print(f'[{entry["lote_id"]}] Generando {entry["cantidad"]} CIF — {entry["sistema"]}',flush=True)
    started=time.monotonic()
    # Conservo la salida de consola del modelo para poder diagnosticar fallos.
    # Si el tutor pregunta qué pasó en un lote, este archivo es la evidencia.
    with (dest/'consola_mattergen.txt').open('w',encoding='utf-8',errors='replace') as log:
        process=command_runner(cmd,stdout=log,stderr=subprocess.STDOUT,check=False,env={**os.environ,'WANDB_MODE':'disabled'})
    duration=round(time.monotonic()-started,2)
    # Un comando que termina mal NO se convierte en un éxito ficticio.
    if process.returncode!=0:
        write_json(dest/'fallo.json',{**entry,'fecha':now(),'codigo_salida':process.returncode,'segundos':duration})
        raise RuntimeError(f'MatterGen falló en {entry["lote_id"]} (código {process.returncode}); consultar consola_mattergen.txt')
    # Antes de dar el lote por bueno compruebo el ZIP producido por MatterGen.
    if not arch.exists():raise RuntimeError(f'Falta generated_crystals_cif.zip en {dest}')
    n=count_cif(arch)
    if n!=entry['cantidad']:raise RuntimeError(f'{entry["lote_id"]}: se esperaban {entry["cantidad"]}, hay {n} CIF')
    # Solo registro OK cuando hay tantos CIF como se pidieron.
    # Eso todavía no significa que pasen el filtro científico posterior.
    record={**entry,'estado':'OK','cif_generados':n,'segundos':duration,
            'fecha_fin_utc':now(),'archivo_zip_cif':str(arch.relative_to(root)),
            'objetivos_de_generacion_no_medidos':cond}
    write_json(done,record)
    return record

# Este resumen junta los lotes completados y pendientes.
# Es el contador que vemos en Kaggle como 6/100, 11/100, etc.
def manifest(root,plan,cfg):
    root=Path(root)
    rows=[]
    # Al reanudar paso por los lotes en el mismo orden de siempre.
    # Los completos se verifican y se saltan; los nuevos se generan.
    for e in plan:
        p=root/'lotes'/e['lote_id']/'lote_finalizado.json'
        if p.exists():
            rec=json.loads(p.read_text(encoding='utf-8'))
            zipfile_path=root/'lotes'/e['lote_id']/'generated_crystals_cif.zip'
            if zipfile_path.exists() and count_cif(zipfile_path)==e['cantidad']:
                rows.append(rec);continue
        rows.append({**e,'estado':'PENDIENTE','cif_generados':0})
    done=sum(r.get('cif_generados',0) for r in rows if r['estado'].startswith('OK'))
    out={'fecha_utc':now(),'esquema':cfg['esquema'],'checkpoint':MODEL,
         'muestras_programadas':sum(e['cantidad'] for e in plan),'cif_generados_verificados':done,
         'lotes_totales':len(rows),'lotes_completos':sum(r['estado'].startswith('OK') for r in rows),
         'estado':'COMPLETADO' if done==100 else 'PARCIAL',
         'nota':'Objetivo energy_above_hull=0.05 es condicion generativa; no estabilidad calculada',
         'lotes':rows}
    write_json(root/'manifest_generacion.json',out)
    # El CSV de lotes es fácil de abrir en Excel y también queda vinculado al TFM.
    with (root/'registro_lotes.csv').open('w',newline='',encoding='utf-8') as f:
        cols=['lote_id','sistema','familia','indice_lote','cantidad','semilla','estado','cif_generados','segundos']
        w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore');w.writeheader();w.writerows(rows)
    return out

# Este punto de entrada conecta configuración, generación y checkpoints.
# No hay que lanzar a mano 21 comandos separados.
def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--salida',required=True)
    ap.add_argument('--mattergen-exe',default=os.environ.get('MATTERGEN_EXECUTABLE','mattergen-generate'))
    ap.add_argument('--max-lotes',type=int,default=0);ap.add_argument('--checkpoint-zip',default='')
    args=ap.parse_args()
    cfg=load_config(args.config);root=Path(args.salida);root.mkdir(parents=True,exist_ok=True)
    plan=plan_batches(cfg);cmd_count=0
    try:
        for e in plan:
            check=root/'lotes'/e['lote_id']/'lote_finalizado.json'
            # Limitar cantidad de NUEVOS lotes; los ya completados no consumen el límite.
            if check.exists():
                # Verifica de nuevo el ZIP; nunca confía únicamente en la marca de finalizado.
                run_one(e,cfg,root,args.mattergen_exe)
                continue
            if args.max_lotes and cmd_count>=args.max_lotes:break
            cmd_count+=1
            run_one(e,cfg,root,args.mattergen_exe)
            result=manifest(root,plan,cfg)
            print(f'Progreso verificado: {result["cif_generados_verificados"]}/100',flush=True)
            # El lote queda guardado directamente en TFM_RESULTADOS/generacion.
            # NO subimos nada a Kaggle Datasets: la prioridad ahora es terminar.
            # Tampoco empaqueto cada lote: el ZIP único se crea al final.
        result=manifest(root,plan,cfg)
        print(json.dumps({'estado':result['estado'],'cif_generados':result['cif_generados_verificados']},ensure_ascii=False))
        return 0
    # Si falla, dejo manifestado el avance real para intentar recuperarlo después.
    except Exception as exc:
        manifest(root,plan,cfg)
        print('ERROR GENERACIÓN:',exc,file=sys.stderr)
        if args.checkpoint_zip:
            from exportar_kaggle import pack
            pack(root.parent,args.checkpoint_zip)
        return 1
if __name__=='__main__':sys.exit(main())

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/lanzar_mattergen.py
#!/usr/bin/env python3
"""Lanzador reproducible del modelo oficial MatterGen v1.0.3.

La CLI oficial 1.0.3 no expone --seed; se inicializan las RNGs de Python,
NumPy y PyTorch antes de llamar a la API de generación. Los resultados CUDA
pueden ser no deterministas pese a fijar la semilla.
"""
import argparse
import json
import random
import numpy as np
import torch
from mattergen.scripts.generate import main


# Aquí preparo una sola llamada de generación: un lote, una semilla
# y unas condiciones. Es el puente entre nuestro programa y MatterGen oficial.
def run():
    p=argparse.ArgumentParser()
    p.add_argument('--output-path',required=True)
    p.add_argument('--pretrained-name',required=True)
    p.add_argument('--batch-size',type=int,required=True)
    p.add_argument('--seed',type=int,required=True)
    p.add_argument('--conditioning-json',required=True)
    p.add_argument('--guidance',type=float,required=True)
    p.add_argument('--record-trajectories',choices=['true','false'],default='false')
    args=p.parse_args()
    # Fijo las semillas de Python, NumPy y PyTorch.
    # No prometo reproducibilidad bit a bit en cualquier GPU, pero sí un procedimiento claro.
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
    # La condición de composición se lee desde el JSON que prepara el plan.
    # No la escribo a mano en cada ejecución para evitar despistes.
    conditioning=json.loads(args.conditioning_json)
    if not isinstance(conditioning,dict):
        raise ValueError('conditioning-json debe ser un objeto')
    print('[MATTERGEN] semilla:',args.seed,'condiciones:',conditioning,flush=True)
    # Esta llamada sí ejecuta el generador original de Microsoft.
    # Nuestro proyecto controla el proceso, pero no sustituye su algoritmo.
    main(output_path=args.output_path,pretrained_name=args.pretrained_name,
         batch_size=args.batch_size,num_batches=1,
         properties_to_condition_on=conditioning,
         diffusion_guidance_factor=args.guidance,
         record_trajectories=(args.record_trajectories=='true'))


if __name__=='__main__':
    run()

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/instalar_kaggle.py
#!/usr/bin/env python3
"""Instalador MatterGen Kaggle v4.4: CUDA y fuentes oficiales completas.

No modifica el kernel de Kaggle. Aísla los índices para impedir el conflicto
requests/pymatgen/wandb al resolver contra el índice de PyTorch.
La ejecución real en Kaggle debe verificarse externamente.
"""
from __future__ import annotations
import os, shutil, subprocess, sys
from pathlib import Path

# Fijo las fuentes de los paquetes porque mezclar índices fue uno de
# los problemas encontrados al probar la instalación anterior.
TORCH_INDEX='https://download.pytorch.org/whl/cu118'
PYG_WHEELS='https://data.pyg.org/whl/torch-2.2.0+cu118.html'
PYPI_INDEX='https://pypi.org/simple'
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ENVS=ROOT/'tfm_envs'


# Kaggle hereda algunas variables del entorno. Las limpio para evitar que
# una instalación busque dependencias en un repositorio que no corresponde.
def env_uv():
    """Evita que índices de CUDA anteriores contaminen la resolución actual."""
    env=os.environ.copy()
    for k in ('UV_INDEX','UV_INDEX_URL','UV_EXTRA_INDEX_URL','UV_DEFAULT_INDEX',
              'UV_FIND_LINKS','UV_INDEX_STRATEGY','PIP_INDEX_URL','PIP_EXTRA_INDEX_URL',
              'PIP_FIND_LINKS'):
        env.pop(k,None)
    # Kaggle exporta un backend inline del kernel. El venv separado no tiene
    # matplotlib_inline; un proceso no interactivo debe usar Agg.
    env['MPLBACKEND']='Agg'
    env['UV_LINK_MODE']='copy'
    env['UV_INDEX_STRATEGY']='first-index'
    return env


# Esta función pequeña ejecuta las instalaciones y detiene todo si hay un fallo.
# No tendría sentido generar materiales con medio entorno instalado.
def run(args,**kwargs):
    args=list(map(str,args))
    print('>', ' '.join(args),flush=True)
    subprocess.run(args,check=True,env=env_uv(),**kwargs)


# PyTorch es la base de los cálculos en GPU. Uso la versión CUDA
# que resulta compatible con las dependencias de MatterGen de este proyecto.
def comando_torch(uv,py_mg):
    return [uv,'pip','install','--python',py_mg,
            'torch==2.2.1+cu118','torchvision==0.17.1+cu118',
            'torchaudio==2.2.1+cu118','--index-url',TORCH_INDEX]


# PyTorch Geometric necesita extensiones compiladas compatibles.
# Si se descargan ruedas equivocadas, la GPU puede fallar al empezar.
def comando_pyg(uv,py_mg):
    # Las extensiones nativas deben proceder únicamente de ruedas PyG para
    # torch 2.2 y cu118: evita compilarlas desde fuentes en la T4.
    return [uv,'pip','install','--python',py_mg,'--no-index','--no-deps',
            '--only-binary',':all:','--find-links',PYG_WHEELS,
            'pyg_lib','torch_scatter','torch_sparse','torch_cluster']


# Separo PyPI del índice CUDA para evitar conflictos de versiones.
# Aquí aprendí que instalar todo con el mismo índice puede dar errores extraños.
def comando_mattergen(uv,py_mg):
    # CLAVE: no pasar --extra-index-url ni --index URL de PyTorch.
    # PyTorch y PyG ya están instalados; MatterGen usa solo PyPI para
    # resolver requests>=2.32, pymatgen, wandb y demás dependencias.
    return [uv,'pip','install','--python',py_mg,
            'mattergen==1.0.3','numpy<2','setuptools<81',
            '--index-url',PYPI_INDEX]


# Primero compruebo que Kaggle realmente tenga una GPU disponible.
# El portátil local no tiene NVIDIA, por eso este cálculo va en la nube.
def comprobar_gpu_kernel():
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError('Kaggle no tiene PyTorch en el kernel. Selecciona GPU T4 x2.') from exc
    if not torch.cuda.is_available():
        raise RuntimeError('GPU no disponible: Settings > Accelerator > GPU T4 x2. Reinicia sesión.')
    print('[GPU Kaggle]',torch.cuda.get_device_name(0),'(',round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB )',flush=True)


# No basta con que pip diga «instalado»: reviso CUDA y versiones
# desde el propio intérprete que utilizará MatterGen.
def comprobar_entorno(py_mg):
    script=(
        'import importlib.metadata as m,torch,numpy,torch_scatter,torch_sparse,torch_cluster,mattergen; '
        'assert torch.cuda.is_available(), "CUDA no disponible para MatterGen"; '
        'assert torch.__version__.startswith("2.2.1+cu118"),torch.__version__; '
        'assert int(numpy.__version__.split(".")[0])<2,numpy.__version__; '
        'assert m.version("mattergen")=="1.0.3",m.version("mattergen"); '
        'print("[OK] MatterGen GPU comprobado:",torch.__version__,torch.cuda.get_device_name(0))'
    )
    run([py_mg,'-c',script])


# Este arreglo es importante: la rueda publicada omitía archivos de
# configuración; por eso usamos el repositorio oficial en un tag concreto.
def instalar_fuente_oficial(uv, py_mg):
    """La rueda PyPI 1.0.3 omite sampling_conf y gemnet-dT.json.

    Usamos el repositorio oficial fijado al tag v1.0.3. Instalación editable
    SIN dependencias para preservar el torch cu118 comprobado y lo ya instalado.
    """
    repo=ROOT/'tfm_mattergen_fuente_oficial_v1_0_3'
    if not (repo/'.git').is_dir():
        if repo.exists():
            raise RuntimeError(f'Hay una carpeta incompleta en {repo}; revisarla antes de continuar')
        env=env_uv()
        env['GIT_LFS_SKIP_SMUDGE']='1'  # HuggingFace descargará el checkpoint al requerirlo.
        cmd=['git','clone','--depth','1','--single-branch','--branch','v1.0.3',
             'https://github.com/microsoft/mattergen.git',str(repo)]
        print('>', ' '.join(cmd),flush=True)
        subprocess.run(cmd,check=True,env=env)
    # Sin estas configuraciones, MatterGen no sabe cómo iniciar el muestreo.
    if not (repo/'sampling_conf/default.yaml').is_file():
        raise RuntimeError('El repositorio oficial no incluye sampling_conf/default.yaml')
    if not (repo/'mattergen/common/gemnet/gemnet-dT.json').is_file():
        raise RuntimeError('El repositorio oficial no incluye gemnet-dT.json')
    run([uv,'pip','install','--python',py_mg,'--no-deps','-e',str(repo),
         '--index-url',PYPI_INDEX])
    print('[OK] Código oficial MatterGen v1.0.3 editable: recursos completos',flush=True)


# Compruebo expresamente los archivos que faltaban en la instalación v4.3.
# Es mejor localizar el problema aquí que después de esperar la GPU.
def comprobar_archivos_configuracion(py_mg):
    script=(
        'from pathlib import Path; '
        'from mattergen.common.utils.globals import DEFAULT_SAMPLING_CONFIG_PATH; '
        'import mattergen; '
        'from hydra import initialize_config_dir, compose; '
        'p=Path(DEFAULT_SAMPLING_CONFIG_PATH); '
        'q=Path(mattergen.__file__).resolve().parent/"common/gemnet/gemnet-dT.json"; '
        'assert (p/"default.yaml").is_file(),f"Falta {p}/default.yaml"; '
        'assert q.is_file(),f"Falta {q}"; '
        'print("[OK] sampling_conf:",p); '
        'print("[OK] GemNet:",q)'
    )
    run([py_mg,'-c',script])


# Instalo en dos entornos independientes, uno para generar y otro para analizar.
# Así evitamos que las bibliotecas científicas se pisen entre sí.
def install():
    comprobar_gpu_kernel()
    uv=shutil.which('uv')
    if not uv:
        run([sys.executable,'-m','pip','install','uv'])
        uv=shutil.which('uv') or str(Path.home()/'.local/bin/uv')
    if not Path(uv).is_file() and not shutil.which(uv):
        raise RuntimeError('No se ha encontrado uv después de instalarlo')
    run([uv,'python','install','3.10'])
    mg=ENVS/'mattergen';analisis=ENVS/'analisis'
    for path in (mg,analisis):
        if not (path/'bin/python').is_file():
            run([uv,'venv',str(path),'--python','3.10','--seed'])
    py_mg=str(mg/'bin/python');py_a=str(analisis/'bin/python')
    # El orden de instalación está pensado para fijar primero CUDA
    # y después añadir MatterGen sin reemplazar ese núcleo.
    print('[1/4] PyTorch CUDA 11.8: índice exclusivo de PyTorch',flush=True)
    run(comando_torch(uv,py_mg))
    print('[2/4] Extensiones PyTorch Geometric: ruedas compiladas cu118',flush=True)
    run(comando_pyg(uv,py_mg))
    print('[3/4] MatterGen y librerías generales: SOLO PyPI',flush=True)
    run(comando_mattergen(uv,py_mg))
    instalar_fuente_oficial(uv, py_mg)
    comprobar_entorno(py_mg)
    comprobar_archivos_configuracion(py_mg)
    exe=mg/'bin/mattergen-generate'
    if not exe.is_file():
        raise RuntimeError('MatterGen no instaló el ejecutable mattergen-generate')
    # La segunda parte necesita pandas, matminer, pymatgen y scikit-learn
    # para trabajar con datos y validar los CIF.
    print('[4/4] Dependencias del análisis',flush=True)
    run([uv,'pip','install','--python',py_a,'-r',Path(__file__).parent/'requirements_analisis.txt',
         '--index-url',PYPI_INDEX])
    run([py_a,'-c','import numpy,pandas,sklearn,pymatgen,matminer,mp_api,matplotlib;print("[OK] Análisis disponible")'])
    print('[OK] INSTALACIÓN LISTA. El proceso automático sigue con los lotes nuevos hasta 100 CIF.',flush=True)

if __name__=='__main__':
    install()

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/preparar_materiales.py
"""Definiciones de consulta y clasificación para datos de referencia Materials Project.

No contiene datos ficticios ni implementaciones alternativas incompatibles.
El pipeline de Kaggle se coordina desde ``analizar_kaggle.py``.
"""

# Estas son las columnas de Materials Project que necesitaremos.
# Guardo los nombres aquí para no repetirlos por todo el programa.
FIELDS = [
    "material_id", "formula_pretty", "energy_above_hull",
    "formation_energy_per_atom", "density", "band_gap",
    "volume", "nsites", "elements", "chemsys", "is_stable",
]
# Clasifico por elementos presentes; es una etiqueta de organización,
# no una prueba de tipo de electrolito.
FAMILIES = {
    "oxidos": "O", "sulfuros": "S", "fluoruros": "F",
    "cloruros": "Cl", "bromuros": "Br", "yoduros": "I",
    "fosfatos_potenciales": "P",
}


# Convierto la lista de elementos en etiquetas sencillas
# que luego facilitan las tablas y la comparación por familias.
def family(elements):
    els = set(elements)
    tags = []
    if "O" in els: tags.append("oxido")
    if "S" in els: tags.append("sulfuro")
    if els & {"F", "Cl", "Br", "I"}: tags.append("haluro")
    if {"P", "O"} <= els: tags.append("fosfato")
    return ";".join(tags) if tags else "otra"

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/modelos_tfm.py
"""Predictores reproducibles con validación por fórmula y descriptores Magpie.

Las predicciones proceden solo de la composición química: nunca sustituyen una
medición, ni el cálculo de DFT para un cristal particular.
"""
from __future__ import annotations
import json
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Mantengo una semilla fija para que la selección y el entrenamiento
# sean comparables al repetir el procedimiento.
SEED = 42


# Comparo cuatro enfoques, no doy por bueno un modelo sin evaluarlo.
# Los transformadores de datos viven dentro de cada pipeline.
def estimadores():
    return {
        'KNN': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsRegressor(n_neighbors=3)),
        'Random Forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestRegressor(n_estimators=120, random_state=SEED, n_jobs=-1)),
        'Gradient Boosting': make_pipeline(SimpleImputer(strategy='median'), GradientBoostingRegressor(random_state=SEED)),
        'MLP': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPRegressor(hidden_layer_sizes=(40, 20), max_iter=450, early_stopping=True, random_state=SEED))
    }


# Magpie traduce una fórmula química a números que entiende el modelo.
# Aquí NO usamos el resultado que queremos predecir como entrada.
def descriptores_composicion(formulas):
    """Magpie de composition únicamente. No emplear target/ID/descriptores derivados del target."""
    from pymatgen.core import Composition
    from matminer.featurizers.composition import ElementProperty
    # Los descriptores se calculan igual para MP y para los candidatos generados.
    f = ElementProperty.from_preset('magpie')
    rows = []
    # Si una fórmula no se interpreta, marcamos sus descriptores como ausentes
    # para que el análisis decida si puede continuar.
    for formula in formulas:
        try:
            c = Composition(str(formula))
            values = f.featurize(c)
            rows.append(values)
        except Exception:
            rows.append([float('nan')] * len(f.feature_labels()))
    vals = pd.DataFrame(rows, columns=f.feature_labels())
    return vals.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)


# Antes de entrenar quito IDs duplicados y filas sin objetivo conocido.
# El recorte opcional de filas se deja documentado con una semilla.
def preparar_entrenamiento(df, target, max_rows=0):
    from pymatgen.core import Composition
    x = df[['material_id', 'formula_pretty', target]].copy()
    x[target] = pd.to_numeric(x[target], errors='coerce')
    x = x.dropna(subset=['formula_pretty', target]).drop_duplicates('material_id')
    if target == 'energy_above_hull': x = x[x[target] >= 0]
    if target == 'band_gap': x = x[x[target] >= 0]
    x = x.sort_values('material_id').reset_index(drop=True)
    # Este muestreo evita cargas excesivas, pero reduce el conjunto:
    # por eso se debe mencionar en la metodología.
    if max_rows and len(x) > max_rows:
        # Muestreo reproducible; para el estudio final debe documentarse el recorte.
        x = x.sample(n=int(max_rows), random_state=SEED).sort_values('material_id').reset_index(drop=True)
    if len(x) < 30: raise ValueError(f'No hay suficientes registros válidos para {target} ({len(x)} < 30)')
    try:
        groups=x['formula_pretty'].map(lambda s: Composition(str(s)).reduced_formula).astype(str)
    except Exception as exc:
        raise ValueError('Hay fórmulas no válidas en la base de entrenamiento') from exc
    return x, groups


# La evaluación se realiza por fórmulas distintas con GroupKFold.
# Esta separación es más exigente que dividir filas al azar sin mirar duplicados.
def entrenar_target(df, target, out, num_folds=5, max_rows=0):
    work, groups = preparar_entrenamiento(df, target, max_rows)
    if groups.nunique() < num_folds: raise ValueError('Número insuficiente de fórmulas distintas para validación agrupada')
    # Primero obtengo las variables y después descarto columnas vacías.
    features = descriptores_composicion(work.formula_pretty)
    valid_cols = list(features.columns[features.notna().any(axis=0)])
    if len(valid_cols) < 2: raise ValueError('No hay suficientes descriptores Magpie disponibles')
    X = features[valid_cols]
    y = work[target].to_numpy(dtype=float)
    cv = GroupKFold(n_splits=num_folds)
    evaluations=[]
    # MAE, RMSE y R² se calculan sobre pliegues no vistos durante el ajuste.
    # Después podré comparar modelos en la Tabla 14.
    for name, model in estimadores().items():
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            res=cross_validate(model, X, y, cv=cv, groups=groups, scoring={
                'MAE':'neg_mean_absolute_error','RMSE':'neg_root_mean_squared_error','R2':'r2'},n_jobs=1,error_score='raise')
        evaluations.append(dict(variable_objetivo=target, modelo=name,
                                MAE=float(-np.mean(res['test_MAE'])),
                                RMSE=float(-np.mean(res['test_RMSE'])),
                                R2=float(np.mean(res['test_R2'])),
                                validacion='GroupKFold por formula reducida',
                                filas_entrenamiento=len(y),grupos=groups.nunique()))
    table=pd.DataFrame(evaluations).sort_values('RMSE').reset_index(drop=True)
    # Elijo el modelo con menor RMSE medio; es una regla explícita,
    # no el que casualmente produjo la figura más bonita.
    champion=str(table.iloc[0].modelo)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        oof=cross_val_predict(clone(estimadores()[champion]), X, y, groups=groups,cv=cv,n_jobs=1)
    predictions=pd.DataFrame(dict(material_id=work.material_id, formula_pretty=work.formula_pretty,
                                  target=target,real=y,predicho_oof=oof))
    # Solo después de validar entreno el modelo final con todos los datos.
    # Lo guardo para estimar las propiedades de los CIF nuevos.
    fitted=clone(estimadores()[champion]).fit(X,y)
    joblib.dump({'modelo':fitted,'features':valid_cols,'target':target,
                 'metodo':'Magpie composicion sola','model_version':'TFM_HITO3_2026_1'},out/f'modelo_{target}.joblib')
    predictions.to_csv(out/f'predicciones_oof_{target}.csv',index=False)
    table.to_csv(out/f'comparacion_modelos_{target}.csv',index=False)
    return table, predictions, features, work, valid_cols


# La misma transformación Magpie se aplica a los nuevos candidatos.
# Si alguna fórmula no tiene descriptores suficientes, lo notifico.
def predecir_modelos(formulas, resultados):
    formulas=list(formulas)
    X=descriptores_composicion(formulas)
    output={}
    for target in ('energy_above_hull','band_gap'):
        job=joblib.load(Path(resultados)/f'modelo_{target}.joblib')
        xx=X.reindex(columns=job['features'])
        if xx.isna().all(axis=1).any():
            bad=np.flatnonzero(xx.isna().all(axis=1))
            raise ValueError(f'Formulas sin descriptores: indices {bad.tolist()}')
        output[target]=job['modelo'].predict(xx)
    return output

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/graficos_tfm.py
"""Tablas/gráficos del TFM, derivados exclusivamente de CSV reales del pipeline."""
from pathlib import Path
import numpy as np
import pandas as pd
import os
os.environ['MPLBACKEND']='Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# Esta figura enseña la distribución real de energía en Materials Project.
# No sustituimos la muestra por números simulados.
def grafico_20(df, out):
    values=pd.to_numeric(df.energy_above_hull,errors='coerce').dropna()
    if values.empty: raise ValueError('No hay energy_above_hull para la Figura 20')
    # Guardo la figura como PNG para poder insertarla directamente en la memoria.
    fig,ax=plt.subplots(figsize=(7,5))
    ax.hist(values,bins=min(35,max(6,int(np.sqrt(len(values))))),edgecolor='white')
    ax.set_xlabel('Energy above hull (eV/átomo) · Materials Project')
    ax.set_ylabel('Frecuencia');ax.set_title('Figura 20. Energía en el conjunto de referencia')
    fig.tight_layout();fig.savefig(out/'figura_20_distribucion_energia_MP.png',dpi=160);plt.close(fig)


# Pongo en el eje X los valores MP y en Y las predicciones fuera de pliegue.
# La diagonal ayuda a ver cuánto se aparta el modelo de lo esperado.
def figura_28(oof, out, target):
    x=oof['real'].to_numpy();y=oof['predicho_oof'].to_numpy()
    fig,ax=plt.subplots(figsize=(6,6))
    ax.scatter(x,y,s=18,alpha=.65)
    lo,hi=min(x.min(),y.min()),max(x.max(),y.max());ax.plot([lo,hi],[lo,hi],'k--')
    ax.set_xlabel(f'{target}: valor de referencia MP')
    ax.set_ylabel(f'{target}: estimación fuera de muestra')
    ax.set_title('Figura 28. Real vs predicho (según objetivo)')
    fig.tight_layout();fig.savefig(out/f'figura_28_real_vs_predicho_{target}.png',dpi=160);plt.close(fig)


# PCA reduce muchos descriptores a dos ejes para visualizarlos.
# La posición no debe interpretarse como una propiedad electroquímica.
def figuras_23_24(candidatos, resultados, out):
    # PCA sobre descriptores químico-composicionales, no atribuir representatividad electroquímica.
    from modelos_tfm import descriptores_composicion
    feats=descriptores_composicion(candidatos.formula_pretty)
    if len(feats)<4: return {'estado':'omitidas','causa':'menos de 4 candidatos'}
    feats=feats.loc[:,feats.notna().any()]
    if feats.shape[1]<2: return {'estado':'omitidas','causa':'menos de 2 variables Magpie'}
    # Normalizo las variables antes del PCA: si no, una escala grande
    # podría dominar los ejes sin ser más importante científicamente.
    X=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(feats))
    coords=PCA(n_components=2).fit_transform(X)
    table=pd.DataFrame({'material_id':candidatos.material_id,'PC1':coords[:,0],'PC2':coords[:,1]})
    table.to_csv(out/'pca_candidatos.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,5));ax.scatter(coords[:,0],coords[:,1],s=24,alpha=.75)
    ax.set_xlabel('PC1');ax.set_ylabel('PC2');ax.set_title('Figura 23. PCA de candidatos por composición Magpie')
    fig.tight_layout();fig.savefig(out/'figura_23_PCA_candidatos.png',dpi=160);plt.close(fig)
    # Pruebo varios números de grupos en K-Means y comparo silhouette.
    # Es una exploración, no una clasificación química definitiva.
    ks=range(2,min(7,len(coords)-1)+1)
    scores=[]
    for k in ks:
        lab=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(coords)
        if len(set(lab))>1:
            scores.append((k,silhouette_score(coords,lab)))
    if not scores: return {'estado':'PCA creado; clustering omitido','causa':'sin agrupación válida'}
    # Escogemos el k mejor puntuado de los probados; queda registrado
    # para explicar por qué se muestran esos grupos.
    k=max(scores,key=lambda p:p[1])[0]
    labels=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(coords)
    table['cluster']=labels;table.to_csv(out/'pca_y_clusters_candidatos.csv',index=False)
    pd.DataFrame(scores,columns=['k','silhouette']).to_csv(out/'silhouette_k.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,5));ax.scatter(coords[:,0],coords[:,1],s=24,c=labels,cmap='viridis')
    ax.set_xlabel('PC1');ax.set_ylabel('PC2');ax.set_title(f'Figura 24. Agrupación K-Means de candidatos (k={k})')
    fig.tight_layout();fig.savefig(out/'figura_24_KMeans_candidatos.png',dpi=160);plt.close(fig)
    return {'estado':'PCA y clustering generados','k':k,'silhouette':dict(scores)}

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/ranking_tfm.py
"""Evaluación multicriterio HONESTA del electrolito.

Dos productos diferentes:
- ranking_exploratorio: estabilidad y band gap predichos, y novedad de fórmula;
  no certifica propiedades funcionales de electrolitos.
- ranking_integral: solo con las SEIS propiedades documentadas con fuente y
  con pesos aprobados por quien realiza el TFM.
"""
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Estos son los criterios que motivan la investigación.
# No todos tienen un dato calculado: distinguirlo es clave en el TFM.
CRITERIOS_TFM = [
    ('estabilidad_termodinamica','Baja energía sobre la envolvente (eV/átomo)','Filtro primario', 'Energía estimada por ML/comparación MP'),
    ('conductividad_ionica','Alta conductividad de Li+ (S/cm)','Criterio principal','NO calculada por MatterGen ni por MP summary'),
    ('estabilidad_electroquimica','Ventana electroquímica amplia (V)','Criterio principal','NO calculada por band gap'),
    ('compatibilidad_electrodos','Compatibilidad química/interfacial','Complementario','Requiere cálculos/experimentos de interfaces'),
    ('propiedades_mecanicas','Comportamiento mecánico adecuado','Complementario','Requiere evaluaciones mecánicas'),
    ('novedad','Diversidad y ausencia de duplicados','Control de generación','Novedad de fórmula no demuestra novedad estructural'),
]

# La matriz deja explícito qué propiedades podemos evaluar de verdad.
def guardar_matriz_criterios(out: Path):
    out.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([{'criterio':a,'objetivo':b,'funcion_tfm':c,'estado_de_dato':d} for a,b,c,d in CRITERIOS_TFM]).to_csv(out/'tabla_11_criterios_tfm.csv',index=False)


# Como la energía y el band gap tienen unidades diferentes, paso
# a una escala común 0-1 antes de sumarlos.
def normalizar(serie, sentido):
    x=pd.to_numeric(serie,errors='coerce')
    if x.isna().any() or not np.isfinite(x.to_numpy()).all():
        raise ValueError(f'Criterio con valores no numéricos/ausentes: {serie.name}')
    if sentido not in ('min','max'):
        raise ValueError('sentido debe ser min o max')
    rng=float(x.max()-x.min())
    # Si todos sacan lo mismo en un criterio, doy 0,5 a todos.
    # Así esa propiedad no favorece arbitrariamente a nadie.
    if np.isclose(rng,0):
        # Una propiedad constante no discrimina; asignar 0.5 a todos y documentarlo.
        return pd.Series(0.5,index=x.index,dtype=float)
    val=(x-x.min())/rng
    return (1-val if sentido=='min' else val).astype(float)


# La puntuación suma las propiedades normalizadas según los pesos.
# Compruebo que sumen 1 para que el ranking tenga sentido.
def clasificar(df, criterios, id_col='material_id'):
    if df.empty: raise ValueError('No se puede clasificar un conjunto vacío')
    if id_col not in df: raise ValueError(f'Falta ID {id_col}')
    if df[id_col].duplicated().any(): raise ValueError('Candidatos con IDs duplicados')
    if len(criterios)<2: raise ValueError('La evaluación multicriterio requiere >=2 criterios')
    pesos=[float(c['peso']) for c in criterios]
    if any(w<0 or not np.isfinite(w) for w in pesos) or not np.isclose(sum(pesos),1,atol=1e-8):
        raise ValueError('Los pesos deben ser no negativos y sumar 1')
    ranked=df.copy().reset_index(drop=True)
    columnas=[]
    # Cada criterio puede premiar el máximo o el mínimo;
    # por ejemplo, en la energía sobre hull interesa un valor menor.
    for c in criterios:
        columna=c['columna']
        if columna not in ranked: raise ValueError('Falta columna de criterio: '+columna)
        dst='norm_'+columna
        ranked[dst]=normalizar(ranked[columna],c['sentido'])
        columnas.append(dst)
    # Ordeno de mayor a menor y uso el ID como desempate reproducible.
    ranked['puntuacion']=ranked[columnas].to_numpy()@np.asarray(pesos)
    ranked=ranked.sort_values(['puntuacion',id_col],ascending=[False,True],kind='mergesort').reset_index(drop=True)
    ranked.insert(0,'posicion',np.arange(1,len(ranked)+1))
    return ranked, columnas


# Cambio uno por uno los pesos para ver cuánto se mueve la clasificación.
# Si el ranking cambia mucho, debemos reconocer esa limitación.
def sensibilidad(ranked, criterios, cols, out: Path, etiqueta: str):
    ids=ranked.material_id.astype(str).tolist()
    base=np.array([float(c['peso']) for c in criterios])
    scenarios={'Base':base}
    # Aquí doy un 50 % más de peso a cada propiedad de forma individual
    # y renormalizo para seguir sumando 1.
    for idx,c in enumerate(criterios):
        v=base.copy()
        v[idx]*=1.5
        v=v/v.sum()
        scenarios['Priorizar '+c['columna']]=v
    data={'material_id':ids}
    # Guardo la posición de cada candidato bajo cada escenario,
    # no solo una imagen: así se puede auditar la sensibilidad.
    for k,w in scenarios.items():
        score=ranked[cols].to_numpy()@w
        order=np.argsort(-score,kind='stable')
        ranks=np.empty(len(order),dtype=int);ranks[order]=np.arange(1,len(order)+1)
        data[k]=ranks
    sens=pd.DataFrame(data)
    sens.to_csv(out/f'analisis_sensibilidad_{etiqueta}.csv',index=False)
    top=min(5,len(ids))
    fig,ax=plt.subplots(figsize=(9,5.3))
    for i in range(top): ax.plot(list(scenarios),[sens[s].iloc[i] for s in scenarios],marker='o',label=ids[i])
    ax.invert_yaxis(); ax.set_ylabel('Posición (1 = mejor)');ax.set_title(f'Figura 30. Sensibilidad de ranking {etiqueta}')
    ax.tick_params(axis='x',rotation=30)
    ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/f'figura_30_sensibilidad_{etiqueta}.png',dpi=160);plt.close(fig)


# Este ranking usa tres indicadores disponibles. No lo llamamos
# ranking completo de electrolitos porque faltan propiedades esenciales.
def ranking_exploratorio(candidatos,config:dict,out:Path):
    out.mkdir(parents=True,exist_ok=True)
    scfg=config['seleccion']
    criteria=[
        dict(columna='energy_above_hull_predicha_eV_atom',sentido='min',peso=float(scfg['pesos_exploratorios']['energy_above_hull_predicha'])),
        dict(columna='band_gap_predicho_eV',sentido='max',peso=float(scfg['pesos_exploratorios']['band_gap_predicho_proxy'])),
        dict(columna='novedad_formula',sentido='max',peso=float(scfg['pesos_exploratorios']['novedad_formula']))
    ]
    df=candidatos.copy()
    # Los filtros opcionales quedan en la configuración.
    # Si ninguno los supera, no invento una clasificación vacía.
    if scfg.get('max_energy_above_hull_predicha_ev_atom') is not None:
        df=df[df['energy_above_hull_predicha_eV_atom']<=float(scfg['max_energy_above_hull_predicha_ev_atom'])]
    if scfg.get('min_band_gap_ev') is not None:
        df=df[df['band_gap_predicho_eV']>=float(scfg['min_band_gap_ev'])]
    if df.empty: raise ValueError('Ningún candidato supera filtros; no se crea ranking')
    # Estos pesos (0,65 / 0,25 / 0,10) son exploratorios
    # y hay que justificarlos, no interpretarlos como una verdad física.
    ranked, cols=clasificar(df,criteria)
    ranked['tipo_ranking']='EXPLORATORIO — NO APTITUD COMO ELECTROLITO DEMOSTRADA'
    ranked.to_csv(out/'tabla_16_ranking_EXPLORATORIO.csv',index=False)
    pd.DataFrame([{**c,'etiqueta':'Solo exploratorio','justificacion':'La conductividad iónica y la ventana electroquímica no se han medido ni calculado'} for c in criteria]).to_csv(out/'tabla_15_pesos_EXPLORATORIOS.csv',index=False)
    fig,ax=plt.subplots(figsize=(9,5))
    top=ranked.head(5)
    top.set_index('material_id')[cols].plot(kind='bar',ax=ax,ylim=(0,1))
    ax.set_ylabel('Valor normalizado');ax.set_title('Figura 29. Indicadores exploratorios (no propiedades funcionales completas)')
    ax.tick_params(axis='x',rotation=25);fig.tight_layout();fig.savefig(out/'figura_29_propiedades_exploratorias.png',dpi=160);plt.close(fig)
    sensibilidad(ranked,criteria,cols,out,'exploratorio')
    return ranked


COLUMNAS_COMPLETAS=[
    ('energy_above_hull_validada_eV_atom','min'),
    ('conductividad_ionica_S_cm','max'),
    ('ventana_electroquimica_V','max'),
    ('compatibilidad_interfacial_puntuacion_0_1','max'),
    ('propiedades_mecanicas_puntuacion_0_1','max'),
    ('novedad_formula','max'),
]

# He previsto un ranking integral solo por si llegamos a tener
# validaciones adicionales con metodología y referencia verificables.
def ranking_integral(candidatos, archivo_validaciones, pesos, out):
    """Solo datos aportados por un estudio con fuente verificable. No inventar.

    Requiere columnas de procedencia por material: metodo, referencia.
    La energía puede ser estimación por DFT/MLFF, documentada en procedencia.
    """
    if not archivo_validaciones: return False
    inp=Path(archivo_validaciones)
    if not inp.is_file(): raise FileNotFoundError(inp)
    # El fichero externo debe identificar material, método y fuente.
    # Sin esa trazabilidad, no le daría valor como evidencia en la defensa.
    v=pd.read_csv(inp)
    required=['material_id','metodo','referencia']+[k for k,_ in COLUMNAS_COMPLETAS if k!='novedad_formula']
    missing=[c for c in required if c not in v]
    if missing: raise ValueError('Validaciones científicas incompletas; faltan: '+', '.join(missing))
    if v.material_id.duplicated().any(): raise ValueError('Hay duplicados en datos validados')
    if v['metodo'].isna().any() or v['referencia'].isna().any() or (v['referencia'].astype(str).str.strip()=='').any():
        raise ValueError('Cada validación requiere metodología y referencia')
    if not pesos or set(pesos)!={k for k,_ in COLUMNAS_COMPLETAS}:
        raise ValueError('Pesos del ranking integral deben definirse explícitamente para los seis criterios')
    # Los cuatro campos funcionales se inicializan en NaN en el fichero de
    # trazabilidad; al incorporar validaciones externas se sustituyen, no
    # se mezclan creando columnas _x/_y.
    cols_aportadas=[k for k,_ in COLUMNAS_COMPLETAS if k!='novedad_formula']
    tabla_base=candidatos.drop(columns=cols_aportadas,errors='ignore')
    # Uno los datos por ID para evitar asignar propiedades al cristal equivocado.
    df=tabla_base.merge(v,on='material_id',how='inner',validate='one_to_one')
    if df.empty: raise ValueError('No hay coincidencias entre IDs de candidatos y validaciones')
    # No reemplazamos sin aviso ausencias por valores inventados.
    # Si falta una medida, se detiene: no relleno huecos con cero
    # porque eso alteraría artificialmente el resultado.
    required_numeric=[k for k,_ in COLUMNAS_COMPLETAS]
    if df[required_numeric].isna().any().any(): raise ValueError('Hay validaciones numéricas ausentes para algunos candidatos')
    for c in required_numeric:
        df[c]=pd.to_numeric(df[c],errors='coerce')
    if df[required_numeric].isna().any().any() or not np.isfinite(df[required_numeric].to_numpy(dtype=float)).all():
        raise ValueError('Las validaciones deben contener números finitos en todos los criterios')
    if (df['conductividad_ionica_S_cm']<=0).any(): raise ValueError('La conductividad debe ser > 0 S/cm')
    if (df['energy_above_hull_validada_eV_atom']<0).any(): raise ValueError('Energy above hull no puede ser negativa')
    if (df['ventana_electroquimica_V']<0).any(): raise ValueError('Ventana electroquímica no puede ser negativa')
    for k in ('compatibilidad_interfacial_puntuacion_0_1','propiedades_mecanicas_puntuacion_0_1'):
        if not pd.to_numeric(df[k]).between(0,1).all(): raise ValueError(f'{k}: exige puntuación definida en [0,1]')
    criteria=[{'columna':k,'sentido':sentido,'peso':pesos[k]} for k,sentido in COLUMNAS_COMPLETAS]
    ranked, cols=clasificar(df,criteria)
    ranked['tipo_ranking']='INTEGRAL CON DATOS APORTADOS; SEGUIR LIMITACIONES METODOLÓGICAS'
    ranked.to_csv(out/'tabla_16_ranking_integral_CON_DATOS.csv',index=False)
    pd.DataFrame(criteria).to_csv(out/'tabla_15_pesos_integrales.csv',index=False)
    sensibilidad(ranked,criteria,cols,out,'integral')
    return True

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/analizar_kaggle.py
#!/usr/bin/env python3
"""Evaluación científica de CIF realmente generados; jamás inventa propiedades faltantes."""
from __future__ import annotations
import argparse, json, os, shutil, sys, traceback, zipfile
from collections import Counter, defaultdict
from datetime import datetime,timezone
from pathlib import Path
import numpy as np
import pandas as pd
os.environ['MPLBACKEND']='Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from generar_kaggle import load_config, manifest, plan_batches, write_json, now
from modelos_tfm import entrenar_target, predecir_modelos
from graficos_tfm import grafico_20, figura_28, figuras_23_24
from ranking_tfm import ranking_exploratorio, guardar_matriz_criterios
from preparar_materiales import FIELDS, FAMILIES, family


# El conjunto de referencia viene de Materials Project, no de filas inventadas.
# Lo descargo una vez y lo guardo para poder revisar qué datos se usaron.
def get_reference(cfg, out):
    dest=out/'01_datos';dest.mkdir(parents=True,exist_ok=True)
    cleanpath=dest/'materials_limpios.csv'
    # Si ya tenemos datos descargados, evitamos repetir consultas a la API.
    # Esto ahorra tiempo y mantiene el mismo conjunto entre reintentos.
    if cleanpath.exists():
        clean=pd.read_csv(cleanpath)
        if len(clean)>30:
            raw=pd.read_csv(dest/'materials_descargados.csv')
            print(f'[DATOS] Reutilizando {len(clean)} entradas de MP guardadas',flush=True)
            return clean,len(raw),len(raw)-len(raw.drop_duplicates('material_id'))
    # La clave entra mediante una variable de entorno, desde los Secrets de Kaggle.
    # Nunca debe escribirse aquí ni guardarse en GitHub.
    key=os.environ.get('MP_API_KEY','').strip()
    if not key:
        raise RuntimeError('Falta MP_API_KEY. Crea el secreto MP_API_KEY en Kaggle antes del análisis; la clave no se guarda en el proyecto.')
    from mp_api.client import MPRester
    rows=[];records=[]
    # Consulto sistemas que contienen litio y los elementos elegidos.
    # La muestra de referencia no representa necesariamente toda Materials Project.
    with MPRester(key) as mpr:
        for elem in FAMILIES.values():
            print('[MP] Descargando',elem,flush=True)
            try:
                docs=list(mpr.materials.summary.search(elements=['Li',elem],fields=FIELDS))
            # Cuando un CIF no pasa, guardo el motivo del rechazo.
            # No borro silenciosamente los errores porque afectan al resultado final.
            except Exception as exc:
                records.append({'elemento':elem,'estado':'ERROR','mensaje':str(exc)[:150]})
                continue
            docs=sorted(docs,key=lambda d:str(d.material_id));take=int(cfg['datos']['max_por_consulta'])
            chosen=docs[:take] if take>0 else docs
            records.append({'elemento':elem,'estado':'OK','recuperados':len(docs),'incluidos':len(chosen)})
            for d in chosen:
                item={k:getattr(d,k,None) for k in FIELDS}
                item['material_id']=str(item['material_id'])
                elems=[str(x) for x in (item.get('elements') or [])]
                item['elements']=';'.join(elems);item['familias']=family(elems)
                rows.append(item)
    write_json(dest/'registro_descarga.json',{'fuente':'Materials Project (MP API), cálculos de referencia',
        'consultas':records,'fecha':now(),'muestreo':'primeros N identificadores ordenados, NO aleatorio'})
    if not rows: raise RuntimeError('MP no devolvió registros; revisar registro_descarga.json')
    # Primero guardo los datos tal como llegaron; luego limpio duplicados.
    # Así puedo explicar de dónde sale el número de la Tabla 12.
    raw=pd.DataFrame(rows);raw.to_csv(dest/'materials_descargados.csv',index=False)
    clean=raw.drop_duplicates('material_id').dropna(subset=['formula_pretty']).copy()
    for col in ['energy_above_hull','band_gap']:
        clean[col]=pd.to_numeric(clean[col],errors='coerce')
    # No entreno con energías ausentes o físicamente negativas.
    # Esta limpieza es sencilla, pero importa para no sesgar el modelo.
    clean=clean[clean.energy_above_hull.notna() & clean.energy_above_hull.ge(0)]
    clean=clean.sort_values('material_id').reset_index(drop=True)
    if len(clean)<60:raise RuntimeError(f'Solo {len(clean)} materiales de referencia; entrenamiento insuficiente')
    clean.to_csv(cleanpath,index=False)
    return clean,len(raw),len(raw)-len(raw.drop_duplicates('material_id'))


# Aquí sí revisamos los CIF de MatterGen. Los contamos, abrimos y filtramos.
# Generar un archivo no es lo mismo que demostrar que el material sirve.
def screening(cfg, root, out, reference):
    from pymatgen.core import Structure, Composition
    from pymatgen.analysis.structure_matcher import StructureMatcher
    data=[];reject=[];unique=defaultdict(list)
    # StructureMatcher ayuda a detectar estructuras muy parecidas.
    # Esto evita presentar dos cristales equivalentes como dos propuestas distintas.
    matcher=StructureMatcher(ltol=0.2,stol=0.3,angle_tol=5)
    # La novedad se compara solo con las fórmulas del subconjunto descargado.
    # No se puede afirmar que una composición es inédita en todo el mundo.
    known=set()
    for v in reference.formula_pretty.dropna().astype(str):
        try:known.add(Composition(v).reduced_formula)
        except Exception:pass
    source=Path(root)/'lotes';filenames=sorted(source.glob('*/generated_crystals_cif.zip'))
    allowed={entry['lote_id']:set(entry['sistema'].split('-')) for entry in plan_batches(cfg)}
    idx=0
    cif_dir=out/'03_mattergen'/'cif_validados';cif_dir.mkdir(parents=True,exist_ok=True)
    # Voy leyendo los ZIP originales, sin fabricar un CSV de candidatos a mano.
    # Cada fila conserva el lote y el fichero CIF de procedencia.
    for archive in filenames:
        batch=archive.parent.name
        if batch not in allowed:raise RuntimeError(f'Lote desconocido: {batch}')
        with zipfile.ZipFile(archive) as z:
            for name in sorted(z.namelist()):
                if not name.lower().endswith('.cif') or name.endswith('/'):continue
                idx+=1;mid=f'MG{idx:05d}';origin=f'{archive.relative_to(root)}::{name}'
                try:
                    content=z.read(name).decode('utf-8',errors='replace')
                    # Pymatgen interpreta el cristal y nos deja medir volumen, sitios
                    # y distancias para realizar filtros geométricos objetivos.
                    struct=Structure.from_str(content,fmt='cif')
                    if not struct.is_ordered:raise ValueError('Ocupaciones atómicas desordenadas')
                    if len(struct)<2 or len(struct)>256:raise ValueError('Número de sitios fuera de [2,256]')
                    chem={x.symbol for x in struct.composition.elements}
                    # Compruebo que aparecen exactamente los elementos solicitados en ese lote.
                    # Si el resultado no corresponde al sistema previsto, lo descarto y explico por qué.
                    if chem!=allowed[batch]:
                        raise ValueError(f'Sistema generado {sorted(chem)} difiere del condicionado {sorted(allowed[batch])}')
                    if struct.volume<=0 or not np.isfinite(struct.volume):raise ValueError('Volumen no positivo o no finito')
                    d=struct.distance_matrix.copy();np.fill_diagonal(d,np.inf)
                    if np.nanmin(d)<0.60:raise ValueError('Distancia interatómica <0.60 Å (filtro geométrico)')
                    formula=struct.composition.reduced_formula
                    # Entre estructuras de igual fórmula no quiero contar duplicados evidentes.
                    if any(matcher.fit(s,struct) for s in unique[formula]):
                        raise ValueError('Estructura repetida (StructureMatcher dentro de igual fórmula)')
                    unique[formula].append(struct)
                    (cif_dir/f'{mid}.cif').write_text(content,encoding='utf-8')
                    # La fila aceptada conserva la procedencia y NO se etiqueta como ensayo experimental.
                    # Lo importante es que podamos volver desde la tabla al CIF original.
                    data.append({'material_id':mid,'lote_id':batch,'formula_pretty':formula,
                        'formula_celda':struct.composition.formula.replace(' ',''),
                        'chemical_system':'-'.join(sorted(chem)), 'num_sitios':len(struct),
                        'volumen_A3':round(struct.volume,5),'novedad_formula':int(formula not in known),
                        'novedad_estructural_verificada':False,'origen_cif_zip':str(archive.relative_to(root)),
                        'nombre_cif_en_zip':name,'ruta_cif_valido':f'03_mattergen/cif_validados/{mid}.cif',
                        'origen_datos':'CIF generado MatterGen; no ensayo experimental'})
                # Si el análisis falla, anoto la fase y la traza, sin declarar éxito.
                except Exception as exc:
                    reject.append({'material_id':mid,'lote_id':batch,'origen':origin,'motivo':str(exc)[:250]})
    # Si no hay cristales reales, la evaluación se detiene.
    # No hay gráficos de resultados construidos con datos de ejemplo.
    if not idx:raise RuntimeError('No se han encontrado CIF de MatterGen')
    folder=out/'03_mattergen'
    pd.DataFrame(data,columns=['material_id','lote_id','formula_pretty','chemical_system','novedad_formula']).to_csv(folder/'candidatos_generados_validos.csv',index=False)
    pd.DataFrame(reject,columns=['material_id','lote_id','origen','motivo']).to_csv(folder/'candidatos_descartados.csv',index=False)
    if not data:raise RuntimeError('No hay CIF válidos; revisar candidatos_descartados.csv')
    df=pd.DataFrame(data)
    write_json(folder/'recuento_filtros.json',{'CIF_leidos':idx,'CIF_aceptados':len(data),
      'CIF_descartados':len(reject),'regla':'Composición exacta, orden, 2-256 sitios, volumen, distancia >=0.60 A, duplicados estructurales por fórmula'})
    # Estos recuentos por sistema darán la figura de aceptados y rechazados.
    sys_counts=[]
    for s in cfg['generacion']['sistemas']:
        expected=s['sistema'];batch_ids=[p['lote_id'] for p in plan_batches(cfg) if p['sistema']==expected]
        n=sum(sum(r['lote_id']==b for r in data) for b in batch_ids)
        nreject=sum(sum(r['lote_id']==b for r in reject) for b in batch_ids)
        sys_counts.append({'sistema':expected,'familia':s['familia'],'planificados':s['numero'],
                           'CIF_encontrados':n+nreject,'aceptados':n,'descartados':nreject})
    tab=pd.DataFrame(sys_counts);tab.to_csv(folder/'tabla_filtros_por_sistema.csv',index=False)
    fig,ax=plt.subplots(figsize=(9,4.8))
    pos=np.arange(len(tab));ax.bar(pos,tab.aceptados,label='Aceptados');
    ax.bar(pos,tab.descartados,bottom=tab.aceptados,label='Descartados')
    ax.set_xticks(pos,tab.sistema);ax.set_ylabel('CIF generados');ax.set_title('Resultados reales: filtros por sistema químico')
    ax.legend();fig.tight_layout();fig.savefig(folder/'figura_filtros_por_sistema.png',dpi=160);plt.close(fig)
    return df,idx,len(reject)


# Esta es la secuencia completa del análisis. La ejecuto después
# de generar, porque necesita CIF reales y datos de referencia.
def main():
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);p.add_argument('--generados',required=True)
    p.add_argument('--salida',required=True);p.add_argument('--permitir-parcial',action='store_true')
    args=p.parse_args();cfg=load_config(args.config);gen=Path(args.generados);out=Path(args.salida)
    out.mkdir(parents=True,exist_ok=True)
    manifest_path=out/'manifiesto_analisis.json'
    tracking={'fecha_inicio':now(),'estado':'EN_CURSO','total_objetivo':100,'fases':[]}
    write_json(manifest_path,tracking)
    try:
        generation=manifest(gen,plan_batches(cfg),cfg)
        # Por defecto exijo las 100 muestras. Si hay menos, solo analizo
        # con --permitir-parcial para que la limitación quede explícita.
        if generation['cif_generados_verificados'] != 100 and not args.permitir_parcial:
            raise RuntimeError(f"Generación incompleta: {generation['cif_generados_verificados']}/100. Completar lotes o usar --permitir-parcial etiquetando la muestra.")
        (out/'configuracion_utilizada.json').write_text(Path(args.config).read_text(encoding='utf-8'),encoding='utf-8')
        tracking['generacion']={k:generation[k] for k in ('estado','muestras_programadas','cif_generados_verificados','lotes_completos')}
        tracking['fases'].append('Generación revisada')
        # Primero preparo los datos de referencia; sin ellos no hay aprendizaje supervisado.
        reference,raw,dups=get_reference(cfg,out)
        models=out/'02_modelos';models.mkdir(exist_ok=True)
        grafico_20(reference,models)
        tracking['fases'].append('Referencia Materials Project descargada/verificada')
        metrics=[];features=[]
        # Entreno dos objetivos distintos: energía sobre hull y band gap.
        # Ninguno representa directamente la conductividad del litio.
        for target in cfg['modelo_predictivo']['targets']:
            scores,oof,_,_,features=entrenar_target(reference,target,models,
                num_folds=int(cfg['modelo_predictivo']['num_folds']),
                max_rows=int(cfg['modelo_predictivo']['max_filas_entrenamiento']))
            metrics.append(scores);figura_28(oof,models,target)
        compare=pd.concat(metrics,ignore_index=True)
        compare.to_csv(models/'tabla_14_modelos_predictivos.csv',index=False)
        for target in cfg['modelo_predictivo']['targets']:
            part=compare[compare.variable_objetivo==target]
            fig,ax=plt.subplots(figsize=(8,4.5))
            ax.barh(part.modelo,part.RMSE)
            unidad='eV/átomo' if target=='energy_above_hull' else 'eV'
            ax.set_xlabel(f'RMSE ({unidad})')
            ax.set_title(f'Comparación real de error: {target}')
            fig.tight_layout();fig.savefig(models/f'figura_RMSE_{target}.png',dpi=160);plt.close(fig)
        pd.DataFrame([{'Caracteristica':'Materiales MP descargados (con solapes)','Valor':raw},
           {'Caracteristica':'Duplicados MP por ID','Valor':dups},
           {'Caracteristica':'Materiales MP limpios','Valor':len(reference)},
           {'Caracteristica':'Columnas Magpie válidas','Valor':len(features)},
           {'Caracteristica':'Semilla','Valor':cfg['semilla']},
           {'Caracteristica':'Targets supervisados','Valor':'energy_above_hull, band_gap'},
           {'Caracteristica':'Conductividad iónica calculada','Valor':'NO'},
           {'Caracteristica':'Ventana electroquímica calculada','Valor':'NO'}]
          ).to_csv(out/'01_datos'/'tabla_12_resumen.csv',index=False)
        tracking['fases'].append('Modelos entrenados y evaluados GroupKFold')
        # Tras evaluar los modelos, filtro los materiales realmente generados.
        candidates,n_raw,n_rej=screening(cfg,gen,out,reference)
        tracking['fases'].append('CIF analizados/filtrados')
        # Las predicciones se basan en composición química Magpie.
        # No equivalen a un cálculo DFT nuevo para cada estructura.
        pred=predecir_modelos(candidates.formula_pretty,models)
        candidates['energy_above_hull_predicha_eV_atom']=pred['energy_above_hull']
        candidates['band_gap_predicho_eV']=pred['band_gap']
        candidates['procedencia_predicciones']='ML composicional Magpie, no DFT individual ni medición'
        # Estas propiedades son esenciales para un electrolito, pero aquí no
        # tenemos medidas válidas. Las dejo vacías en lugar de inventarlas.
        for col in ('conductividad_ionica_S_cm','ventana_electroquimica_V',
                    'compatibilidad_interfacial_puntuacion_0_1','propiedades_mecanicas_puntuacion_0_1'):
            candidates[col]=np.nan
        selection=out/'04_seleccion';selection.mkdir(exist_ok=True)
        candidates.to_csv(selection/'tabla_13_propiedades_y_trazabilidad.csv',index=False)
        guardar_matriz_criterios(selection)
        # PCA y K-Means sirven para explorar similitudes composicionales.
        # No demuestran que los materiales transporten iones de litio.
        cluster=figuras_23_24(candidates,models,models)
        # El ranking final de esta fase es orientativo.
        # Una buena puntuación no convierte al candidato en electrolito validado.
        ranking=ranking_exploratorio(candidates,cfg,selection)
        tracking['fases'].append('Ranking exploratorio y sensibilidad calculados')
        summary={'CIF_planificados':100,'CIF_generados_y_verificados':n_raw,
          'CIF_aceptados':len(candidates),'CIF_descartados':n_rej,
          'Candidatos_en_ranking_exploratorio':len(ranking),'PCA_clustering':cluster,
          'R2_y_MAE_RMSE':'Ver tabla_14_modelos_predictivos.csv',
          'conductividad_ionica':'NO evaluada','ventana_electroquimica':'NO evaluada',
          'estabilidad_energia':'Predicha con modelo de composición; no verificada con DFT',
          'ranking':'Exploratorio, NO desempeño electroquímico demostrado',
          'distribucion':'25 por sistema (diseño exploratorio, no muestra óptima)'}
        write_json(out/'resumen_numerico.json',summary)
        lines=['# Resultados reales del experimento',f'Fecha UTC: {now()}',
               f'Muestras planificadas: 100',f'CIF encontrados: {n_raw}',
               f'CIF aceptados tras filtros: {len(candidates)}',f'CIF descartados: {n_rej}',
               f'Entradas ranking exploratorio: {len(ranking)}','',
               '## Métricas (GroupKFold por fórmula, media por pliegue)',compare.to_markdown(index=False),
               '', '## Resultados que NO se pueden afirmar',
               '- No hay conductividad iónica calculada ni medida.',
               '- Band gap electrónico no equivale a ventana electroquímica.',
               '- La energía predicha por composición no es energía DFT del cristal generado.',
               '- Estar cerca del objetivo generativo de energía no demuestra estabilidad.',
               '- Ningún candidato se considera electrolito confirmado.',
               '', '## PCA y clustering',json.dumps(cluster,ensure_ascii=False,default=str)]
        # Dejo un resumen legible para trasladar las cifras al TFM
        # y para preparar las respuestas a las preguntas del tutor.
        (out/'RESUMEN_RESULTADOS_TFM.md').write_text('\n'.join(lines),encoding='utf-8')
        tracking.update({'estado':'COMPLETADO','fecha_fin':now(),'resumen':summary})
        print('ANÁLISIS COMPLETADO',json.dumps(summary,ensure_ascii=False,default=str),flush=True)
        return 0
    except Exception as exc:
        tracking.update({'estado':'ERROR','fecha_fin':now(),'error':str(exc),'traza':traceback.format_exc()[-5000:]})
        print('ERROR ANÁLISIS:',exc,file=sys.stderr)
        return 1
    finally:write_json(manifest_path,tracking)

if __name__=='__main__':sys.exit(main())

In [ ]:
%%writefile /kaggle/working/TFM_KAGGLE_100/ejecutar_todo_kaggle.py
#!/usr/bin/env python3
"""Controlador de una ejecución nueva del TFM en Kaggle: versión 4.8.

En lugar de ir celda por celda, este controlador ejecuta la instalación,
la generación, el análisis y la exportación de resultados.

He eliminado los respaldos remotos, los tokens de Kaggle Datasets y también
los ZIP intermedios. Todos los CIF se escriben directamente en la carpeta
de resultados y el ZIP final se crea al acabar. Si falla algo, intento
conservar lo disponible en un ZIP parcial, señalando el error.

ATENCIÓN: la carpeta es temporal. Hay que descargar los resultados antes de
que Kaggle cierre o reinicie la sesión.
"""
from __future__ import annotations

import io
import json
import os
import subprocess
import sys
import traceback
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from exportar_kaggle import pack

# Dejamos las rutas escritas una única vez para evitar NameError entre celdas.
BASE = Path('/kaggle/working')
PROYECTO = BASE / 'TFM_KAGGLE_100'
RESULTADOS = BASE / 'TFM_RESULTADOS'
GENERACION = RESULTADOS / 'generacion'
ANALISIS = RESULTADOS / 'analisis'
ZIP_FINAL = BASE / 'Resultados_TFM_Kaggle_100.zip'
MATTERGEN = BASE / 'tfm_envs/mattergen/bin/mattergen-generate'
PY_ANALISIS = BASE / 'tfm_envs/analisis/bin/python'


def registrar(etapa: str, mensaje: str, estado: str) -> None:
    """Registro simple, sin inventar éxito cuando hay una fase incompleta."""
    ruta = RESULTADOS / 'estado_ejecucion_v48.json'
    ruta.parent.mkdir(parents=True, exist_ok=True)
    contenido = {
        'fecha_utc': datetime.now(timezone.utc).isoformat(),
        'etapa': etapa, 'estado': estado, 'mensaje': mensaje,
        'nota': 'Generar CIF no equivale a validar un electrolito sólido',
    }
    temporal = ruta.with_suffix('.tmp')
    temporal.write_text(json.dumps(contenido, indent=2, ensure_ascii=False), encoding='utf-8')
    temporal.replace(ruta)
    print(f'[{estado}] {etapa}: {mensaje}', flush=True)


def lanzar(*orden: object, entorno: dict | None = None) -> None:
    """Si algo falla, muestro el fallo real y no sigo como si nada."""
    print('INICIANDO:', Path(str(orden[0])).name, flush=True)
    subprocess.run([str(x) for x in orden], check=True,
                   cwd=str(PROYECTO), env=entorno)


def ejecutar_desde_cero() -> None:
    # Si ya existen cristales, no los borro. La primera celda se ocupa
    # de preparar un entorno limpio y proteger datos reales previos.
    if GENERACION.exists() and any(GENERACION.iterdir()):
        raise RuntimeError('La carpeta de generación no está vacía. No se mezcla ni sobrescribe.')
    if ZIP_FINAL.exists():
        raise RuntimeError('Hay un ZIP previo en la sesión. No se sobreescribe automáticamente.')
    PROYECTO.mkdir(parents=True, exist_ok=True)
    RESULTADOS.mkdir(parents=True, exist_ok=True)
    os.environ['MPLBACKEND'] = 'Agg'

    # Solo es obligatoria la API de Materials Project. No uso ninguna API de
    # datasets privados de Kaggle, ni siquiera para comprobar su existencia.
    if not os.environ.get('MP_API_KEY', '').strip():
        raise RuntimeError('Falta MP_API_KEY en Kaggle Secrets.')
    registrar('inicio', 'Nueva ejecución con 0/100 CIF; sin respaldos remotos', 'INICIADO')

    # Si la GPU está desactivada, lo sabemos enseguida, no tras instalar todo.
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Activa GPU T4 en Kaggle antes de Run All.')
    print('GPU:', torch.cuda.get_device_name(0), flush=True)

    # Conservo la instalación y el generador de la rama que ya había generado
    # cristales. Lo único que suprimo aquí es el bloqueo del backup remoto.
    registrar('instalación', 'Preparando MatterGen y librerías', 'INICIADO')
    lanzar(sys.executable, PROYECTO / 'instalar_kaggle.py')
    if not MATTERGEN.is_file() or not PY_ANALISIS.is_file():
        raise RuntimeError('Faltan ejecutables después de la instalación.')
    registrar('instalación', 'Instalación verificada', 'OK')

    # Son 100 INTENTOS planeados: uno inicial para probar y después lotes de
    # hasta cinco. El contador avanza solo cuando el ZIP tiene CIF reales.
    registrar('generación', 'Comienza MatterGen desde 0/100', 'INICIADO')
    lanzar(sys.executable, PROYECTO / 'generar_kaggle.py',
           '--config', PROYECTO / 'config_tfm_kaggle.json',
           '--salida', GENERACION,
           '--mattergen-exe', MATTERGEN)

    manifest = json.loads((GENERACION / 'manifest_generacion.json').read_text(encoding='utf-8'))
    n = int(manifest.get('cif_generados_verificados', -1))
    if n != 100 or manifest.get('estado') != 'COMPLETADO':
        raise RuntimeError(f'Generación incompleta: {n}/100 CIF.')
    # Contamos de nuevo en sus ZIP originales, por separado.
    # No depende de la API de Kaggle ni de un checkpoint intermedio.
    from generar_kaggle import count_cif
    archivos = sorted((GENERACION / 'lotes').glob('*/generated_crystals_cif.zip'))
    suma = sum(count_cif(archivo) for archivo in archivos)
    if suma != 100:
        raise RuntimeError(f'Los ZIP de los lotes contienen {suma} CIF, no 100.')
    registrar('generación', '100/100 CIF reales encontrados y contados', 'OK')

    # No confundo la generación con el análisis posterior: disponer de CIF no
    # demuestra conductividad iónica ni estabilidad electroquímica medida.
    registrar('análisis', 'Comienzan modelos, filtros, PCA, gráficos y ranking', 'INICIADO')
    entorno = os.environ.copy()
    entorno['MPLBACKEND'] = 'Agg'
    estado_analisis = 'OK'
    try:
        lanzar(PY_ANALISIS, PROYECTO / 'analizar_kaggle.py',
               '--config', PROYECTO / 'config_tfm_kaggle.json',
               '--generados', GENERACION, '--salida', ANALISIS, entorno=entorno)
    except Exception as error:
        # Si falla Materials Project o un cálculo, conservo los 100 CIF.
        # El ZIP dirá que el análisis fue parcial; no crearé métricas falsas.
        estado_analisis = 'PARCIAL'
        registrar('análisis', f'Incidencia: {type(error).__name__}: {error}', 'AVISO')
    else:
        registrar('análisis', 'Análisis finalizado; revisar sus archivos', 'OK')

    # No consulto Kaggle Datasets ni la API de respaldos. Un solo ZIP final:
    # es lo que necesito para descargarlo y completar el TFM.
    registrar('final', f'CIF=100; análisis={estado_analisis}',
             'OK' if estado_analisis == 'OK' else 'PARCIAL')
    pack(RESULTADOS, ZIP_FINAL)
    print('PROCESO FINALIZADO.', flush=True)
    print('CARPETA DE RESULTADOS:', RESULTADOS, flush=True)
    print('ZIP FINAL:', ZIP_FINAL, flush=True)
    print('ATENCIÓN: descarga el ZIP antes de cerrar o perder la sesión.', flush=True)


def main() -> None:
    try:
        ejecutar_desde_cero()
    except Exception as error:
        # No dejo que el último fallo oculte el progreso anterior.
        # Si existen datos, intento dejar su ZIP local actualizado.
        print('ERROR REAL:', type(error).__name__, str(error), flush=True)
        traceback.print_exc()
        try:
            registrar('error', f'{type(error).__name__}: {error}', 'ERROR')
            # Dejo un ZIP incluso si el proceso es parcial. Al abrirlo se
            # verá el estado ERROR o PARCIAL y los lotes que existan.
            if RESULTADOS.exists():
                pack(RESULTADOS, ZIP_FINAL)
        except Exception as empaquetado:
            print('AVISO: no se pudo exportar el ZIP parcial:', empaquetado, flush=True)
        raise


if __name__ == '__main__':
    main()

## 2. Ejecutar el experimento completo

Esta celda se ejecuta **automáticamente** con `Run All`. Primero instala las dependencias, después genera las estructuras y finalmente intenta calcular tablas, gráficos y ranking.

**No hay llamadas de respaldo remoto.** Si la generación falla, el controlador dejará la información disponible en la carpeta de resultados y tratará de crear un ZIP parcial. Si el análisis falla, mantendrá los CIF generados y declarará la fase como parcial.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Lanzo el controlador general. No requiere ir ejecutando celdas manualmente.
comando = [sys.executable, '/kaggle/working/TFM_KAGGLE_100/ejecutar_todo_kaggle.py']
proceso = subprocess.run(comando, cwd='/kaggle/working/TFM_KAGGLE_100', env=os.environ.copy(), check=False)

# Pase lo que pase, muestro las carpetas reales para poder recuperarlas.
raiz = Path('/kaggle/working')
carpeta = raiz / 'TFM_RESULTADOS'
archivo = raiz / 'Resultados_TFM_Kaggle_100.zip'
print('\n===== DÓNDE ESTÁN LOS RESULTADOS =====')
print('Carpeta:', carpeta)
print('Existe:', carpeta.exists())
print('ZIP:', archivo)
print('ZIP disponible:', archivo.is_file())
if archivo.is_file():
    print('Tamaño del ZIP:', archivo.stat().st_size, 'bytes')
    print('Descárgalo desde el explorador de archivos de Kaggle mientras la sesión siga activa.')
if carpeta.exists():
    print('Contenido principal:', [p.name for p in sorted(carpeta.iterdir())])
if proceso.returncode != 0:
    raise RuntimeError('La ejecución terminó con una incidencia. Revisa el error anterior y conserva el ZIP parcial si está disponible.')